# Comparing Funding Across Sources

A key use case for `award_pynder` is comparing how different funders invest in a topic. This notebook demonstrates searching across multiple sources for a common keyword and analyzing the results.

## Research Question

*How do major US funders invest in "climate change" research?*

In [ ]:
from award_pynder import search_awards

# Search across multiple federal and private sources
results = search_awards(
    keywords="climate change",
    sources=["nsf", "nih", "mellon", "sloan"],
    from_date="2023-01-01",
    to_date="2023-06-01",
    verbose=False,
)

print(f"Total grants found: {len(results)}")

## Grants by source

In [ ]:
import pandas as pd

# Count and total funding by source
if not results.empty:
    # Convert amount to numeric where possible
    results["amount_numeric"] = pd.to_numeric(results["amount"], errors="coerce")

    summary = results.groupby("source").agg(
        grant_count=("id", "count"),
        total_funding=("amount_numeric", "sum"),
        avg_award=("amount_numeric", "mean"),
    ).round(0)

    print(summary.to_string())
else:
    print("No results found")

## Top-funded institutions

In [ ]:
if not results.empty:
    top_institutions = (
        results.groupby("institution")
        .agg(
            grant_count=("id", "count"),
            total_funding=("amount_numeric", "sum"),
            sources=("source", lambda x: ", ".join(sorted(set(x)))),
        )
        .sort_values("total_funding", ascending=False)
        .head(15)
    )
    top_institutions

## Exporting results

Save combined results to CSV for further analysis in other tools:

In [ ]:
# Save to CSV (uncomment to run)
# results.to_csv("climate_change_grants_2023.csv", index=False)
# print(f"Saved {len(results)} grants to climate_change_grants_2023.csv")

# Or use the CLI equivalent:
# !award-pynder "climate change" -s nsf,nih,mellon,sloan -f 2023-01-01 -t 2023-06-01 -o climate_change_grants_2023.csv